# Model Training and Evaluation — Solution

**Dataset:** Car fuel consumption in litres per 100 km.

Run from top to bottom. The steps match the exercise; short answers explain the results.

## Libraries and settings

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import cross_val_score

## 1. Compute an error measure

Observed values are 6 and 10; predictions are 5 and 7. Replace the gap with `np.mean(errors ** 2)`. What are the units of RMSE?

In [ ]:
observed = np.array([6, 10])
predicted = np.array([5, 7])
errors = observed - predicted
mse = np.mean(errors ** 2)
print('MSE:', mse, 'RMSE:', np.sqrt(mse))

**Answer:** MSE = 5 and RMSE ≈ 2.24. RMSE has the units of the target: L/100 km.

## 2. Load and split the data

Preparation is provided. Which variables are inputs? Why do we reserve test rows?

In [ ]:
cars = pd.read_csv('../00_Data/car_fuel_consumption_2025.csv')
cars = cars.dropna(subset=['engine_size', 'cylinders', 'combined'])
print(cars[['engine_size', 'cylinders', 'combined']].head())
X = cars[['engine_size', 'cylinders']]
y = cars['combined']  # litres per 100 km
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)
print('Training rows:', len(X_train), 'Test rows:', len(X_test))

**Answer:** Engine size and cylinders are inputs; combined consumption is the target. Test rows represent cars not used for fitting.

## 3. Train a first model

Replace the gap with `LinearRegression()`. Are predictions identical to observed values?

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)
prediction = model.predict(X_test)
print(pd.DataFrame({'Observed': y_test, 'Predicted': prediction}).head().round(2))
print('Test RMSE:', np.sqrt(mean_squared_error(y_test, prediction)))

**Answer:** They differ: the model captures an average relationship and does not use every feature of a car.

## 4. Cross-validation

CV code is provided. Scikit-learn returns negative MSE because its score convention is “larger is better”; the minus sign converts it back. Why are test rows excluded?

In [ ]:
scores = cross_val_score(LinearRegression(), X_train, y_train,
                         cv=5, scoring='neg_mean_squared_error')
fold_rmse = np.sqrt(-scores)
print('Fold RMSE:', fold_rmse.round(2))
print('Mean CV RMSE:', fold_rmse.mean().round(2))

**Answer:** CV evaluates choices using training data, keeping the final test independent.

## 5. Compare model complexity

Run the supplied comparison. Which depth has the smallest CV error? Does smallest training error identify the best model? This is a demonstration; the earlier linear model remains our fixed test example.

In [ ]:
rows = []
for depth in [1, 3, 15]:
    tree = DecisionTreeRegressor(max_depth=depth, random_state=42)
    tree.fit(X_train, y_train)
    train_rmse = np.sqrt(mean_squared_error(y_train, tree.predict(X_train)))
    scores = cross_val_score(tree, X_train, y_train, cv=5,
                             scoring='neg_mean_squared_error')
    rows.append([depth, train_rmse, np.sqrt(-scores).mean()])
results = pd.DataFrame(rows, columns=['Depth', 'Training RMSE', 'CV RMSE'])
print(results.round(2))
results.plot(x='Depth', y=['Training RMSE', 'CV RMSE'], marker='o')
plt.ylabel('RMSE [L/100 km]')
plt.show()

**Answer:** Compare the CV column. A deeper tree may memorise training details. A small training error alone cannot establish good generalisation.

### Jupyter notebook — footer info

Include this information when submitting your notebook.

In [ ]:
import platform
from datetime import datetime
print(platform.platform())
print("Python:", platform.python_version())
print("Date:", datetime.now().isoformat(timespec="seconds"))